In [ ]:
import sys, os
import torch

print("Python:", sys.version.splitlines()[0])
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Working dir:", os.getcwd())

Python: 3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]
Torch: 2.8.0+cu126
CUDA available: True
Working dir: /content


In [ ]:
!pip install -q transformers sentence-transformers faiss-cpu bert-score rouge-score datasets tqdm

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.0 MB/s eta 0:00:00


In [ ]:
!pip install -q --upgrade transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 85.2 MB/s eta 0:00:00


In [ ]:
import os, textwrap

BASE = "rag-summarizer"
os.makedirs(BASE, exist_ok=True)

dirs = [
    os.path.join(BASE, "data", "sample_inputs"),
    os.path.join(BASE, "src"),
    os.path.join(BASE, "eval"),
]

for d in dirs:
    os.makedirs(d, exist_ok=True)

# requirements.txt
req = textwrap.dedent("""\
transformers
sentence-transformers
faiss-cpu
torch
datasets
rouge-score
bert-score
tqdm
""")
open(os.path.join(BASE, "requirements.txt"), "w").write(req)


88

In [ ]:
# src/utils.py
with open(os.path.join(BASE, "src", "utils.py"), "w") as f:
    f.write(textwrap.dedent("""\
from typing import List

def chunk_text(text: str, max_words: int = 200) -> List[str]:
    words = text.split()
    chunks = []
    for i in range(0, len(words), max_words):
        chunks.append(" ".join(words[i:i+max_words]))
    return chunks
"""))


In [ ]:
# src/retrieval.py
with open(os.path.join(BASE, "src", "retrieval.py"), "w") as f:
    f.write(textwrap.dedent("""\
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

class Retriever:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.model = SentenceTransformer(model_name)
        self.index = None
        self.corpus = []

    def build(self, passages: list):
        self.corpus = passages
        embs = self.model.encode(passages, convert_to_numpy=True, show_progress_bar=True)
        faiss.normalize_L2(embs)
        d = embs.shape[1]
        self.index = faiss.IndexFlatIP(d)
        self.index.add(embs)

    def retrieve(self, queries: list, top_k: int = 5):
        q_embs = self.model.encode(queries, convert_to_numpy=True, show_progress_bar=False)
        faiss.normalize_L2(q_embs)
        D, I = self.index.search(q_embs, top_k)
        results = []
        for inds in I:
            results.append([self.corpus[i] for i in inds])
        return results
"""))


In [ ]:
# src/summarizer.py
with open(os.path.join(BASE, "src", "summarizer.py"), "w") as f:
    f.write(textwrap.dedent("""\
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

class Summarizer:
    def __init__(self, model_name='facebook/bart-large-cnn', device=None):
        '''
        Initialize the summarizer with a Hugging Face seq2seq model.
        '''
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        self.model.to(self.device)

    def generate_candidates(self, input_text: str, num_return_sequences=4, max_length=150):
        '''
        Generate multiple summary candidates using sampling (top-k + top-p).
        '''
        inputs = self.tokenizer(
            input_text,
            return_tensors='pt',
            truncation=True,
            max_length=1024
        ).to(self.device)

        outputs = self.model.generate(
            **inputs,
            do_sample=True,       # use sampling for diversity
            top_k=50,             # top-k sampling
            top_p=0.95,           # nucleus sampling
            num_return_sequences=num_return_sequences,
            max_length=max_length,
            early_stopping=True,
            num_beams=1
        )

        return [
            self.tokenizer.decode(
                o, skip_special_tokens=True, clean_up_tokenization_spaces=True
            ) for o in outputs
        ]
"""))


In [ ]:
import os
os.chdir("/content/rag-summarizer")
os.listdir()

from src.summarizer import Summarizer

text = "Google was founded in 1998 by Larry Page and Sergey Brin while PhD students at Stanford University."
summarizer = Summarizer()
candidates = summarizer.generate_candidates(text, num_return_sequences=3, max_length=60)

for i, c in enumerate(candidates):
    print(f"Candidate {i+1}:", c)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['early_stopping', 'length_penalty']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Candidate 1: Google was founded in 1998 by Larry Page and Sergey Brin while PhD students at Stanford University. Google was founded by LarryPage and Sergeybrin while doctoral students at Stanford University. The company was co-founded by Page and Brin in 1997. The founders were still
Candidate 2: Google was founded by Larry Page and Sergey Brin in 1998. The pair were PhD students at Stanford University. Page and Brin are now chief executives of the search giant. Click through the gallery to see other famous Google founders. Share your memories of Google with CNN iReport.
Candidate 3: Google was founded in 1998 by Larry Page and Sergey Brin while PhD students at Stanford University. The search giant now counts over two dozenmillion users across the world. The company also has more than 100,000 Flip flops. They are used by millions of consumers all over the world


In [ ]:
import os, textwrap

BASE = "rag-summarizer"
SRC = os.path.join(BASE, "src")
os.makedirs(SRC, exist_ok=True)   # create src folder if it doesn't exist

with open(os.path.join(SRC, "reranker.py"), "w") as f:
    f.write(textwrap.dedent("""\
import torch
from transformers import pipeline
from bert_score import score as bert_score

class Reranker:
    def __init__(self, nli_model='facebook/bart-large-mnli'):
        device = 0 if torch.cuda.is_available() else -1
        self.nli = pipeline(
            'text-classification',
            model=nli_model,
            device=device,
            return_all_scores=True
        )

    def entailment_score(self, premise: str, hypothesis: str):
        inp = premise + " </s> " + hypothesis
        out = self.nli(inp)[0]
        entail = 0.0
        for d in out:
            if 'ENTAIL' in d['label'].upper():
                entail = d['score']
        return float(entail)

    def score_candidates(self, source: str, candidates: list):
        entail_scores = [self.entailment_score(source, c) for c in candidates]
        P, R, F = bert_score(
            candidates,
            [source]*len(candidates),
            lang='en',
            rescale_with_baseline=True
        )
        final_scores = [0.6*e + 0.4*float(f) for e, f in zip(entail_scores, F)]
        return final_scores
"""))


In [ ]:
import os
os.chdir("/content/rag-summarizer")
os.listdir()

from src.summarizer import Summarizer

text = "Google was founded in 1998 by Larry Page and Sergey Brin while PhD students at Stanford University."
summarizer = Summarizer()
candidates = summarizer.generate_candidates(text, num_return_sequences=3, max_length=60)

for i, c in enumerate(candidates):
    print(f"Candidate {i+1}:", c)


Candidate 1: Larry Page and Sergey Brin were PhDs while studying at Stanford University. Their firm was started in 1998 in memory of former Stanford co-founder Sergey Brins. Page and Brin are still both full professors at Google. Google is one of the most popular search engines in the
Candidate 2: Google was founded in 1998 by Larry Page and Sergey Brin while PhD students at Stanford University. It was sold to Larry Page for $90 million in 1999. It is now worth an estimated $9 million. It has now been valued at $10 billion.
Candidate 3: Google was founded by Google co-founders Larry Page and Sergey Brin in 1998. The two met while PhD students at Stanford University. It is the world's most valuable internet company. The company was founded in the aftermath of the September 11 attacks. The pair have since become some


In [ ]:
import os
os.makedirs("/content/rag-summarizer/src", exist_ok=True)


In [ ]:
import textwrap

with open("/content/rag-summarizer/src/reranker.py", "w") as f:
    f.write(textwrap.dedent("""\
import torch
from transformers import pipeline
from bert_score import score as bert_score

class Reranker:
    def __init__(self, nli_model='facebook/bart-large-mnli'):
        device = 0 if torch.cuda.is_available() else -1
        self.nli = pipeline('text-classification', model=nli_model, device=device, return_all_scores=True)

    def entailment_score(self, premise: str, hypothesis: str):
        inp = premise + " </s> " + hypothesis
        out = self.nli(inp)[0]
        entail = 0.0
        for d in out:
            if 'ENTAIL' in d['label'].upper():
                entail = d['score']
        return float(entail)

    def score_candidates(self, source: str, candidates: list):
        entail_scores = [self.entailment_score(source, c) for c in candidates]
        P, R, F = bert_score(candidates, [source]*len(candidates), lang='en', rescale_with_baseline=True)
        final_scores = [0.6*e + 0.4*float(f) for e, f in zip(entail_scores, F)]
        return final_scores
"""))


In [ ]:
# src/demo.py
with open(os.path.join(BASE, "src", "demo.py"), "w") as f:
    f.write(textwrap.dedent("""\
from src.retrieval import Retriever
from src.summarizer import Summarizer
from src.reranker import Reranker
from src.utils import chunk_text

def summarize_article(text: str):
    if not text.strip():
        print("No text provided.")
        return

    # 1) Chunk text into passages
    passages = chunk_text(text, max_words=120)

    # 2) Build FAISS retriever
    retriever = Retriever()
    retriever.build(passages)

    # 3) Retrieve relevant passages (query = full text)
    retrieved = retriever.retrieve([text], top_k=6)[0]
    evidence = "\\n\\n".join(retrieved)

    # 4) Generate candidate summaries
    summarizer = Summarizer()
    input_for_model = evidence + "\\n\\nSummarize the above in 3 sentences:"
    candidates = summarizer.generate_candidates(input_for_model, num_return_sequences=6, max_length=160)

    # 5) Rerank candidates to pick best
    reranker = Reranker()
    scores = reranker.score_candidates(evidence, candidates)
    best_idx = max(range(len(scores)), key=lambda i: scores[i])

    # 6) Output best summary
    print("\\n=== BEST SUMMARY ===\\n")
    print(candidates[best_idx])
    return candidates[best_idx]

if __name__ == "__main__":
    # Sample text (replace with any article)
    sample_text = \"\"\"Google was founded in 1998 by Larry Page and Sergey Brin while PhD students at Stanford University.
    The company quickly grew and launched products like Gmail, Maps, and Android. It later restructured into Alphabet Inc. in 2015, with Sundar Pichai as CEO of Google.\"\"\"

    summarize_article(sample_text)
"""))


In [ ]:
import os
os.chdir("/content/rag-summarizer")

In [ ]:
from src.summarizer import Summarizer

text = "Google was founded in 1998 by Larry Page and Sergey Brin while PhD students at Stanford University."
summarizer = Summarizer()
candidates = summarizer.generate_candidates(text, num_return_sequences=6, max_length=60)

for i, c in enumerate(candidates):
    print(f"Candidate {i+1}:", c)


Candidate 1: Google was founded by Larry Page and Sergey Brin in 1998. Page and Brin were PhD students at Stanford University. Today the couple run a search engine and start a book company called Google. They have two daughters who make up the 'front office' team at Google.
Candidate 2: Google was founded in 1998 by Larry Page and Sergey Brin. The pair were PhD students at Stanford University. They met on campus while doing their masters in computer science. Click here to follow Google's events on twitter. Watch a live coverage of Google's 20th anniversary on CNN i
Candidate 3: Google was founded in 1998 by Larry Page and Sergey Brin while in undergraduates at Stanford University. Google launched a search engine using e-mail and Google search to aid students in starting up their first Web sites. Google was formed with six employees. It was founded by Google co
Candidate 4: Google was founded in 1998 by Larry Page and Sergey Brin while PhD students at Stanford University. The company no

In [ ]:
from src.reranker import Reranker

reranker = Reranker()
scores = reranker.score_candidates(text, candidates)

# pick the best candidate
best_idx = max(range(len(scores)), key=lambda i: scores[i])
print("\n=== BEST SUMMARY ===\n")
print(candidates[best_idx])

print("\n=== ALL CANDIDATES + SCORES ===\n")
for i, (c, s) in enumerate(zip(candidates, scores)):
    print(f"[{i}] score={s:.4f}\n{c}\n---\n")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



=== BEST SUMMARY ===

Google was founded in 1998 by Larry Page and Sergey Brin while PhD students at Stanford University. The company now employs around 5,000 workers across the world. Page and Brin are described as highly motivated. They have described themselves as 'our team. We're in it together.'

=== ALL CANDIDATES + SCORES ===

[0] score=0.2157
Google was founded by Larry Page and Sergey Brin in 1998. Page and Brin were PhD students at Stanford University. Today the couple run a search engine and start a book company called Google. They have two daughters who make up the 'front office' team at Google.
---

[1] score=0.2137
Google was founded in 1998 by Larry Page and Sergey Brin. The pair were PhD students at Stanford University. They met on campus while doing their masters in computer science. Click here to follow Google's events on twitter. Watch a live coverage of Google's 20th anniversary on CNN i
---

[2] score=0.2096
Google was founded in 1998 by Larry Page and Sergey Brin

In [ ]:
def summarize_article(text):
    summarizer = Summarizer()
    candidates = summarizer.generate_candidates(text, num_return_sequences=6, max_length=150)

    reranker = Reranker()
    scores = reranker.score_candidates(text, candidates)

    # Find the best candidate
    best_idx = scores.index(max(scores))
    best_summary = candidates[best_idx]

    print("=== BEST SUMMARY ===\n")
    print(best_summary)
    print("\n=== ALL CANDIDATES + SCORES ===\n")
    for i, (c, s) in enumerate(zip(candidates, scores)):
        print(f"[{i}] score={s:.4f}\n{c}\n---")

    return best_summary


In [ ]:
article = """
OpenAI develops advanced AI models and platforms to benefit humanity.
Their GPT models are widely used for natural language understanding and generation.
"""
summary = summarize_article(article)


Device set to use cuda:0
/usr/local/lib/python3.12/dist-packages/transformers/pipelines/text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


=== BEST SUMMARY ===

OpenAI develops advanced AI models and platforms to benefit humanity. Their GPT models are widely used for natural language understanding. One of OpenAI's AI projects is to build a language generation system from data. Its research included natural language interpreting and speech recognition. For more information on OpenAI, visit their website.

=== ALL CANDIDATES + SCORES ===

[0] score=0.2127
OpenAI's GPT models are widely used for natural language understanding and generation. Their machine learning algorithms are now being used on a large scale in a variety of applications across businesses. GPT is the first AI technology to be used in a commercial IT service for a major global business.
---
[1] score=0.2583
openAI develops advanced AI models and platforms to benefit humanity. GPT models are widely used for natural language understanding and generation. OpenAI was founded in 1988 and the main focus of the company was to develop self-driving cars. Today the co

In [ ]:
def summarize_article(text, summarizer, reranker, num_candidates=6, max_length=150):
    candidates = summarizer.generate_candidates(text, num_return_sequences=num_candidates, max_length=max_length)
    scores = reranker.score_candidates(text, candidates)
    best_idx = scores.index(max(scores))
    best_summary = candidates[best_idx]

    print("=== BEST SUMMARY ===\n")
    print(best_summary)
    print("\n=== ALL CANDIDATES + SCORES ===\n")
    for i, (c, s) in enumerate(zip(candidates, scores)):
        print(f"[{i}] score={s:.4f}\n{c}\n---")

    return best_summary

In [ ]:
summarizer = Summarizer()
reranker = Reranker()

Device set to use cuda:0
/usr/local/lib/python3.12/dist-packages/transformers/pipelines/text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [ ]:
articles = [
    "Google was founded in 1998 by Larry Page and Sergey Brin while PhD students at Stanford University.",
    "OpenAI develops advanced AI models and platforms to benefit humanity."
]

for article in articles:
    summarize_article(article, summarizer, reranker)


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


=== BEST SUMMARY ===

Google was founded in 1998 by Larry Page and Sergey Brin while PhD students at Stanford University. Larry Page is CEO and Sergeybrin is vice president. Google's Larry Page has been at the company for 20 years. The two were awarded Turing Prize for invention for their work on the Google search engine in 2003.

=== ALL CANDIDATES + SCORES ===

[0] score=0.1275
Larry Page, Sergey Brin founded Google in 1998. In 2002, they unveiled their first search engine. The project was a collaboration between the two Google engineers at Stanford University. It is now one of the biggest tech start-ups in the world. A Google IPO is not unheard of.
---
[1] score=0.2062
Google was founded in 1998 by Larry Page and Sergey Brin while PhD students at Stanford University. Its CEO is Google's former co-founder Sergey Brino. There's an official Google app that the founders use. The company also has more than 10 million employees. "We're still building the business," says Page. "This is mor

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


=== BEST SUMMARY ===

OpenAI develops advanced AI models and platforms to benefit humanity. The company is a project of Google. It has created over 1,000 AI projects and projects worldwide. More info on openAI's projects is available on its website. Back to Mail Online home. Back into the page you came from.

=== ALL CANDIDATES + SCORES ===

[0] score=0.2057
OpenAI develops advanced AI models and platforms to benefit humanity. It's been dubbed the AI company to Watch. In the past it has developed algorithms to analyse data. Now it's training AI in a lab and testing it at real estate offices. In 2013, OpenAI created a computer program to detect humanlike patterns in data.
---
[1] score=0.5866
OpenAI develops advanced AI models and platforms to benefit humanity. The company is a project of Google. It has created over 1,000 AI projects and projects worldwide. More info on openAI's projects is available on its website. Back to Mail Online home. Back into the page you came from.
---
[2] sco

In [ ]:
from src.summarizer import Summarizer
from src.reranker import Reranker

# Initialize models once
summarizer = Summarizer()
reranker = Reranker()

def summarize_article(article):
    # Step 1: Generate multiple candidates
    candidates = summarizer.generate_candidates(article, num_return_sequences=6, max_length=150)

    # Step 2: Score candidates
    scores = reranker.score_candidates(article, candidates)

    # Step 3: Pick the best one
    best_index = scores.index(max(scores))
    best_summary = candidates[best_index]

    return best_summary

# Example usage
articles = [
    "Google was founded in 1998 by Larry Page and Sergey Brin while PhD students at Stanford University.",
    "OpenAI develops advanced AI models and platforms to benefit humanity."
]

for article in articles:
    best_summary = summarize_article(article)
    print("\nArticle:\n", article)
    print("\nBest Summary:\n", best_summary)


Device set to use cuda:0
/usr/local/lib/python3.12/dist-packages/transformers/pipelines/text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Article:
 Google was founded in 1998 by Larry Page and Sergey Brin while PhD students at Stanford University.

Best Summary:
 Google was founded in 1998 by Larry Page and Sergey Brin while PhD students at Stanford University. Google's founder Larry Page first used Google searches to map the Internet. Sergey Brink joined Google later, as a Google executive. The pair began Google as PhD students while in Stanford.


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Article:
 OpenAI develops advanced AI models and platforms to benefit humanity.

Best Summary:
 OpenAI develops advanced AI models and platforms to benefit humanity. The company will also develop next generation software. For more information on OpenAI, visit the company's website. Click to visit the OpenAI website. For the latest on Open AI and Next AI, visit their website.


In [ ]:
!pip install gradio --quiet

import gradio as gr
from src.summarizer import Summarizer
from src.reranker import Reranker

# Initialize models
summarizer = Summarizer()
reranker = Reranker()

def summarize_article_ui(article):
    # Generate candidates
    candidates = summarizer.generate_candidates(article, num_return_sequences=6, max_length=150)

    # Score candidates
    scores = reranker.score_candidates(article, candidates)

    # Pick the best
    best_index = scores.index(max(scores))
    best_summary = candidates[best_index]

    return best_summary

# Create Gradio interface
iface = gr.Interface(
    fn=summarize_article_ui,
    inputs=gr.Textbox(lines=10, placeholder="Paste your article here..."),
    outputs=gr.Textbox(lines=5, placeholder="Best summary will appear here..."),
    title="Rag Summarizer",
    description="Paste any article, and the AI will generate the single best summary for you."
)

# Launch the UI
iface.launch()


Device set to use cuda:0
/usr/local/lib/python3.12/dist-packages/transformers/pipelines/text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://651cd04f8605548437.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


SpaceX, founded by Elon Musk in 2002, is an American aerospace manufacturer and space transport services company. The company aims to reduce space transportation costs and enable the colonization of Mars. SpaceX has developed the Falcon 1, Falcon 9, and Falcon Heavy rockets, along with the Dragon spacecraft family. In 2020, SpaceX became the first private company to send astronauts to the International Space Station. The company continues to innovate in reusable rocket technology and satellite internet via its Starlink project.